In [ ]:
import itertools
import pandas as pd
from pathlib import Path
from datetime import datetime

In [ ]:
KEYWORD_CONFIG = {

    # ========= 广义 object =========
    "object_categories": {

        "camera_optics": [
            "camera", "camera lens", "drone", "gopro", "photography gear"
        ],

        "digital_accessories": [
            "cables", "charger", "power bank", "ssd", "hard drive"
        ],

        "gaming_audio": [
            "controller", "keyboard", "audio gear", "gaming accessories"
        ],

        "tools_equipment": [
            "tools", "equipment", "fragile equipment", "precision tools"
        ],

        "travel_outdoor": [
            "travel gear", "camping gear", "hiking gear", "outdoor equipment"
        ],

        "collectibles_valuables": [
            "watches", "jewelry", "collectibles", "figurines"
        ],

        "beauty_medical": [
            "makeup kit", "beauty devices", "medical devices"
        ]
    },

    # ========= leader用（视频入口） =========

    "new_product_queries": [
        "new gadgets 2026",
        "latest gear release",
        "new equipment unboxing",
        "latest tech and gear"
    ],

    "unboxing_review_queries": [
        "unboxing new gear",
        "gear review",
        "equipment review",
        "first look gadgets"
    ],

    "usage_queries": [
        "gear travel setup",
        "bag setup gear",
        "how i pack my gear",
        "equipment organization"
    ],

    "problem_queries": [
        "how to protect gear",
        "how to store equipment",
        "equipment damaged in bag"
    ],

    # ========= 成员用 =========

    "object_patterns": [
        "{obj} unboxing",
        "{obj} review",
        "{obj} travel setup",
        "{obj} bag setup",
        "how to protect {obj}",
        "{obj} damaged in bag"
    ],

    # ========= 过滤 =========

    "negative_terms": [
        "iphone", "phone case", "smartphone",
        "ipad", "tablet",
        "fashion case", "cute case", "aesthetic case"
    ]
}

In [ ]:
def filter_queries(df, negative_terms):
    df["query_lower"] = df["query"].str.lower()
    mask = ~df["query_lower"].apply(
        lambda x: any(n in x for n in negative_terms)
    )
    return df[mask].drop(columns=["query_lower"]).drop_duplicates()

配置已保存到： ../../data/shared_data/config/keyword_config.json


In [ ]:
def generate_member_queries(config, categories):
    rows = []

    for cat in categories:
        objs = config["object_categories"][cat]

        for obj, pattern in itertools.product(objs, config["object_patterns"]):
            rows.append({
                "category": cat,
                "object": obj,
                "query": pattern.format(obj=obj),
                "type": "object"
            })

    df = pd.DataFrame(rows)
    return filter_queries(df, config["negative_terms"])

dict_keys(['object_categories', 'scenario_queries', 'problem_queries', 'object_scenario_patterns', 'object_problem_patterns', 'product_support_queries', 'global_trends'])


In [ ]:
def generate_leader_queries(config):
    rows = []

    for q in config["new_product_queries"]:
        rows.append({"query": q, "type": "new_product"})

    for q in config["unboxing_review_queries"]:
        rows.append({"query": q, "type": "review"})

    for q in config["usage_queries"]:
        rows.append({"query": q, "type": "usage"})

    for q in config["problem_queries"]:
        rows.append({"query": q, "type": "problem"})

    df = pd.DataFrame(rows)
    return filter_queries(df, config["negative_terms"])

In [ ]:
MEMBER_ASSIGNMENT = {
    "member_1": ["camera_optics"],
    "member_2": ["digital_accessories"],
    "member_3": ["gaming_audio"],
    "member_4": ["tools_equipment"],
    "member_5": ["travel_outdoor", "collectibles_valuables", "beauty_medical"]
}

query 数量： 143


,category,object,query,query_type
0,generic,None,what's in my bag tech,scenario
1,generic,None,everyday carry tech EDC,scenario
2,generic,None,EDC setup gadgets,scenario
3,generic,None,tech travel setup,scenario
4,generic,None,travel gadgets packing,scenario
5,generic,None,camera bag setup,scenario
6,generic,None,drone travel setup,scenario
7,generic,None,photography gear setup,scenario
8,generic,None,minimalist tech setup,scenario
9,generic,None,desk setup tech gadgets,scenario


In [ ]:
output_dir = Path("../data/shared_data/query_assignments")
output_dir.mkdir(exist_ok=True)

today = datetime.now().strftime("%Y%m%d")

all_tables = []

# 成员
for m, cats in MEMBER_ASSIGNMENT.items():
    df = generate_member_queries(KEYWORD_CONFIG, cats)
    df["assigned_to"] = m

    df.to_csv(output_dir / f"{m}_{today}.csv", index=False)
    all_tables.append(df)

# leader
leader_df = generate_leader_queries(KEYWORD_CONFIG)
leader_df["assigned_to"] = "leader"

leader_df.to_csv(output_dir / f"leader_{today}.csv", index=False)

all_tables.append(leader_df)

print("All CSV generated!")

In [ ]:
df_all = pd.concat(all_tables)

dup = df_all.groupby("query")["assigned_to"].nunique().reset_index()
dup = dup[dup["assigned_to"] > 1]

print("重复 query 数量:", len(dup))
dup.head()

In [ ]:
simple_dir = output_dir / "simple"
simple_dir.mkdir(exist_ok=True)

for m in MEMBER_ASSIGNMENT.keys():
    df = pd.read_csv(output_dir / f"{m}_{today}.csv")
    df[["query"]].drop_duplicates().to_csv(
        simple_dir / f"{m}_query_only.csv",
        index=False
    )

leader_df[["query"]].to_csv(
    simple_dir / "leader_query_only.csv",
    index=False
)

print("Simple CSV ready!")